<a href="https://colab.research.google.com/github/Mike-Haldeman30/Flyers-Hockey-Analytics/blob/main/Flyers_Hockey_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Downloading Data for Flyers Shots


In [1]:
import pandas as pd
import matplotlib.pyplot as plt

shots25 = pd.read_csv('/content/drive/MyDrive/Hockey Analytics/shots_2025-current.csv')
Flyersdf = shots25.copy()
Flyersdf = Flyersdf[Flyersdf['teamCode']=='PHI']

#Flyers dataset
Flyersdf = Flyersdf[['goal','xGoal','arenaAdjustedXCordABS', 'arenaAdjustedYCord',
                     'shooterName', 'season', 'period', 'shotType', 'shotRebound',
                     'shotRush', 'shotOnEmptyNet', 'isPlayoffGame', 'isHomeTeam',
                     'homeSkatersOnIce','awaySkatersOnIce', 'teamCode', 'goalieNameForShot',
                     'arenaAdjustedShotDistance', 'shotAngleAdjusted']]

Flyersdf['Shots'] = len(Flyersdf)

Flyersdf.to_csv('flyersCurrent.csv', index=False)

Flyersdf.head()



,goal,xGoal,arenaAdjustedXCordABS,arenaAdjustedYCord,shooterName,season,period,shotType,shotRebound,shotRush,shotOnEmptyNet,isPlayoffGame,isHomeTeam,homeSkatersOnIce,awaySkatersOnIce,teamCode,goalieNameForShot,arenaAdjustedShotDistance,shotAngleAdjusted,Shots
915,0,0.033466,33.0,2.0,Nick Seeler,2025,1,SLAP,0,0,0,0,0,4,4,PHI,Sergei Bobrovsky,56.0,2.045408,3037
920,0,0.024966,64.0,28.0,Nick Seeler,2025,1,SNAP,0,0,0,0,0,5,5,PHI,Sergei Bobrovsky,38.0,48.239700,3037
921,0,0.020649,35.0,-15.0,Jamie Drysdale,2025,1,WRIST,0,0,0,0,0,5,5,PHI,Sergei Bobrovsky,56.0,15.524111,3037
922,0,0.022200,46.0,-29.0,Travis Sanheim,2025,1,WRIST,0,0,0,0,0,5,5,PHI,Sergei Bobrovsky,52.0,33.996459,3037
925,0,0.044712,45.0,-0.0,Nicolas Deslauriers,2025,1,SLAP,0,0,0,0,0,5,5,PHI,Sergei Bobrovsky,44.0,0.000000,3037


Create df for League Avg Compare

In [2]:
team_season = shots25.groupby(['season', 'teamCode']).agg(
    shots    = ('shotID', 'count'),
    goals    = ('goal', 'sum'),
    xG       = ('xGoal', 'sum'),
).reset_index()

team_season['shooting_pct'] = team_season['goals'] / team_season['shots']
team_season['xG_diff']      = team_season['goals'] - team_season['xG']

#high danger shots
df_hd = shots25[
    (shots25['arenaAdjustedXCordABS'] >= 69) &
    (shots25['arenaAdjustedYCord'].abs() <= 22)
]

hd_team_season = df_hd.groupby(['season', 'teamCode']).agg(
    hd_shots = ('shotID', 'count'),
    hd_goals = ('goal', 'sum'),
).reset_index()

hd_team_season['hd_shooting_pct'] = hd_team_season['hd_goals'] / hd_team_season['hd_shots']

team_season = team_season.merge(hd_team_season, on=['season', 'teamCode'], how='left')

#league averages per season
league_avg = team_season.groupby('season').agg(
    league_avg_shots        = ('shots',          'mean'),
    league_avg_goals        = ('goals',          'mean'),
    league_avg_xG           = ('xG',             'mean'),
    league_avg_shooting_pct = ('shooting_pct',   'mean'),
    league_avg_xG_diff      = ('xG_diff',        'mean'),
    league_avg_hd_shots     = ('hd_shots',       'mean'),
    league_avg_hd_goals     = ('hd_goals',       'mean'),
    league_avg_hd_shooting  = ('hd_shooting_pct','mean'),
).reset_index()

league_avg = league_avg.round(2)

print(league_avg)
league_avg.to_csv('league_averages.csv', index=False)

   season  league_avg_shots  league_avg_goals  league_avg_xG  \
0    2025           3292.12            237.31         240.16   

   league_avg_shooting_pct  league_avg_xG_diff  league_avg_hd_shots  \
0                     0.07               -2.85              1245.03   

   league_avg_hd_goals  league_avg_hd_shooting  
0               141.94                    0.11  



Downloading Data for Goalie Analysis
---



In [4]:
import pandas as pd
import numpy as np


df = pd.read_csv('/content/drive/MyDrive/Hockey Analytics/Goalies/2025-Goalies.csv')


df = df[(df['situation'] == 'all') & (df['position'] == 'G')].copy()

#format GAME DATE
df['gameDate'] = pd.to_datetime(df['gameDate'], format='%Y%m%d')

#calculate columns
df['GSAx'] = df['flurryAdjustedxGoals'] - df['goals']
df['SV_pct'] = np.where(df['ongoal'] > 0, 1 - (df['goals'] / df['ongoal']), np.nan)
df['xSV_pct'] = np.where(df['ongoal'] > 0, 1 - (df['flurryAdjustedxGoals'] / df['ongoal']), np.nan)
df['SV_pct_above_expected'] = df['SV_pct'] - df['xSV_pct']

#Danger zone save percentages
df['HD_SV_pct'] = np.where(df['highDangerShots'] > 0, 1 - (df['highDangerGoals'] / df['highDangerShots']), np.nan)
df['MD_SV_pct'] = np.where(df['mediumDangerShots'] > 0, 1 - (df['mediumDangerGoals'] / df['mediumDangerShots']), np.nan)
df['LD_SV_pct'] = np.where(df['lowDangerShots'] > 0, 1 - (df['lowDangerGoals'] / df['lowDangerShots']), np.nan)
df['rebound_control_delta'] = df['rebounds'] - df['xRebounds']
df['freeze_delta'] = df['freeze'] - df['xFreeze']

df['GAA'] = np.where(df['icetime'] > 0, (df['goals'] / df['icetime']) * 3600, np.nan)
df['xGA_per60'] = np.where(df['icetime'] > 0, (df['flurryAdjustedxGoals'] / df['icetime']) * 3600, np.nan)

df = df.sort_values(['playerId', 'gameDate'])
df['GSAx_cumulative'] = df.groupby('playerId')['GSAx'].cumsum()

#making dict for easier agg
agg_dict = {
    'gameId': 'count',
    'icetime': 'sum',
    'goals': 'sum',
    'flurryAdjustedxGoals': 'sum',
    'GSAx': 'sum',
    'ongoal': 'sum',
    'highDangerShots': 'sum',
    'highDangerGoals': 'sum',
    'mediumDangerShots': 'sum',
    'mediumDangerGoals': 'sum',
    'lowDangerShots': 'sum',
    'lowDangerGoals': 'sum',
    'rebounds': 'sum',
    'xRebounds': 'sum',
    'freeze': 'sum',
    'xFreeze': 'sum',
}

season_totals = df.groupby(
    ['playerId', 'name', 'playerTeam', 'season']
).agg(agg_dict).reset_index()

season_totals = season_totals.rename(columns={'gameId': 'games_played'})

#Recalculate all rate stats from season totals
season_totals['SV_pct'] = 1 - (season_totals['goals'] / season_totals['ongoal'])
season_totals['xSV_pct'] = 1 - (season_totals['flurryAdjustedxGoals'] / season_totals['ongoal'])
season_totals['SV_pct_above_expected'] = season_totals['SV_pct'] - season_totals['xSV_pct']
season_totals['HD_SV_pct'] = 1 - (season_totals['highDangerGoals'] / season_totals['highDangerShots'])
season_totals['MD_SV_pct'] = 1 - (season_totals['mediumDangerGoals'] / season_totals['mediumDangerShots'])
season_totals['LD_SV_pct'] = 1 - (season_totals['lowDangerGoals'] / season_totals['lowDangerShots'])
season_totals['GAA'] = (season_totals['goals'] / season_totals['icetime']) * 3600
season_totals['xGA_per60'] = (season_totals['flurryAdjustedxGoals'] / season_totals['icetime']) * 3600
season_totals['rebound_control_delta'] = season_totals['rebounds'] - season_totals['xRebounds']
season_totals['freeze_delta'] = season_totals['freeze'] - season_totals['xFreeze']
season_totals['icetime_minutes'] = (season_totals['icetime'] / 60).round(1)

season_totals_qualified = season_totals[season_totals['icetime'] >= 36000].copy()


#create flyers df for game by game
flyers_gbg = df[df['playerTeam'] == 'PHI'].copy()

#Fill NaN danger zone SV% with 0 only on game-by-game file
cols_to_fill = ['HD_SV_pct', 'MD_SV_pct', 'LD_SV_pct']
flyers_gbg[cols_to_fill] = flyers_gbg[cols_to_fill].fillna(0)

#Flyers season totals
flyers_season = season_totals[season_totals['playerTeam'] == 'PHI'].copy()

#Full league qualified — with Flyers highlight flag
league_season = season_totals_qualified.copy()
league_season['is_flyer'] = (league_season['playerTeam'] == 'PHI').astype(int)


flyers_gbg.to_csv('/content/drive/MyDrive/Hockey Analytics/Goalies/flyers_goalies_gbg.csv', index=False)
flyers_season.to_csv('/content/drive/MyDrive/Hockey Analytics/Goalies/flyers_goalies_season.csv', index=False)
league_season.to_csv('/content/drive/MyDrive/Hockey Analytics/Goalies/league_goalies_season.csv', index=False)



In [6]:
danger_records = []

for _, row in flyers_season.iterrows():
    for zone, shots_col, goals_col in [
        ('Low Danger', 'lowDangerShots', 'lowDangerGoals'),
        ('Medium Danger', 'mediumDangerShots', 'mediumDangerGoals'),
        ('High Danger', 'highDangerShots', 'highDangerGoals')
    ]:
        sv_pct = 1 - (row[goals_col] / row[shots_col]) if row[shots_col] > 0 else None
        danger_records.append({
            'name': row['name'],
            'zone': zone,
            'zone_order': ['Low Danger', 'Medium Danger', 'High Danger'].index(zone) + 1,
            'sv_pct': round(sv_pct, 4) if sv_pct else None,
            'shots': row[shots_col],
            'goals': row[goals_col]
        })

# League average rows
for zone, shots_col, goals_col in [
    ('Low Danger', 'lowDangerShots', 'lowDangerGoals'),
    ('Medium Danger', 'mediumDangerShots', 'mediumDangerGoals'),
    ('High Danger', 'highDangerShots', 'highDangerGoals')
]:
    total_shots = league_season[shots_col].sum()
    total_goals = league_season[goals_col].sum()
    sv_pct = round(1 - (total_goals / total_shots), 4) if total_shots > 0 else None
    danger_records.append({
        'name': 'League Average',
        'zone': zone,
        'zone_order': ['Low Danger', 'Medium Danger', 'High Danger'].index(zone) + 1,
        'sv_pct': sv_pct,
        'shots': total_shots,
        'goals': total_goals
    })

danger_zone_df = pd.DataFrame(danger_records)
danger_zone_df['sv_pct_display'] = (danger_zone_df['sv_pct'] * 100).round(1).astype(str) + '%'

print(danger_zone_df.to_string())

danger_zone_df.to_csv('/content/drive/MyDrive/Hockey Analytics/Goalies/flyers_danger_zones.csv',index=False)

               name           zone  zone_order  sv_pct    shots   goals sv_pct_display
0        Dan Vladar     Low Danger           1  0.9673   1439.0    47.0          96.7%
1        Dan Vladar  Medium Danger           2  0.8750    288.0    36.0          87.5%
2        Dan Vladar    High Danger           3  0.7872    141.0    30.0          78.7%
3     Samuel Ersson     Low Danger           1  0.9472    852.0    45.0          94.7%
4     Samuel Ersson  Medium Danger           2  0.8554    242.0    35.0          85.5%
5     Samuel Ersson    High Danger           3  0.8030     66.0    13.0          80.3%
6   Aleksei Kolosov     Low Danger           1  0.9434     53.0     3.0          94.3%
7   Aleksei Kolosov  Medium Danger           2  0.6000     10.0     4.0          60.0%
8   Aleksei Kolosov    High Danger           3  0.7500      4.0     1.0          75.0%
9    League Average     Low Danger           1  0.9637  74708.0  2709.0          96.4%
10   League Average  Medium Danger         

Getting cumulative sum data for Vlader Trend Line

In [ ]:
vladar_trend = flyers_gbg[flyers_gbg['name'] == 'Dan Vladar'][
    ['gameDate', 'GSAx', 'GSAx_cumulative']
].sort_values('gameDate')

print(vladar_trend.to_string())

       gameDate   GSAx  GSAx_cumulative
8116 2025-10-09  0.737            0.737
8121 2025-10-13  0.356            1.093
8126 2025-10-18  0.184            1.277
8131 2025-10-20  0.215            1.492
8136 2025-10-23  0.398            1.890
8141 2025-10-30  3.199            5.089
8146 2025-11-01 -2.213            2.876
8151 2025-11-04 -1.501            1.375
8156 2025-11-06  2.295            3.670
8161 2025-11-12  0.046            3.716
8166 2025-11-15 -1.518            2.198
8171 2025-11-20  0.891            3.089
8176 2025-11-22  1.336            4.425
8181 2025-11-26  0.900            5.325
8186 2025-11-29  0.408            5.733
8191 2025-12-01 -0.851            4.882
8196 2025-12-09  0.317            5.199
8201 2025-12-11 -0.300            4.899
8206 2025-12-14  1.066            5.965
8211 2025-12-16  1.158            7.123
8216 2025-12-22  0.097            7.220
8221 2025-12-28 -1.153            6.067
8226 2025-12-30  0.027            6.094
8231 2026-01-03  0.400            6.494


Creating df for Defensive data

In [8]:
import pandas as pd
import numpy as np

shots = pd.read_csv('/content/drive/MyDrive/Hockey Analytics/shots_2025.csv')
print(f"Total shots loaded: {len(shots)}")

#Shots against Flyers
shots_against = shots[
    ((shots['homeTeamCode'] == 'PHI') & (shots['teamCode'] != 'PHI')) |
    ((shots['awayTeamCode'] == 'PHI') & (shots['teamCode'] != 'PHI'))
].copy()

print(f"Shots against Flyers: {len(shots_against)}")

#Normalize Coordinates for one end
shots_against['x_map'] = shots_against['arenaAdjustedXCordABS']
shots_against['y_map'] = np.where(
    shots_against['arenaAdjustedXCord'] > 0,
    shots_against['arenaAdjustedYCord'],
    -shots_against['arenaAdjustedYCord']
)

def assign_danger_from_xgoal(xgoal):
    if xgoal >= 0.20:
        return 'High Danger'
    elif xgoal >= 0.07:
        return 'Medium Danger'
    else:
        return 'Low Danger'

#Mapping for Power BI
shots_against['danger_zone'] = shots_against['xGoal'].apply(assign_danger_from_xgoal)
shots_against['danger_order'] = shots_against['danger_zone'].map({
    'Low Danger': 1, 'Medium Danger': 2, 'High Danger': 3
})

def shot_outcome(row):
    if row['goal'] == 1:
        return 'Goal'
    elif row['shotWasOnGoal'] == 1:
        return 'Save'
    else:
        return 'Missed'

shots_against['outcome'] = shots_against.apply(shot_outcome, axis=1)

shot_map_cols = [
    'shotID', 'game_id', 'season',
    'x_map', 'y_map',
    'arenaAdjustedShotDistance', 'shotAngle',
    'shotType', 'danger_zone', 'danger_order',
    'outcome', 'goal', 'xGoal',
    'shotWasOnGoal', 'shotRebound', 'shotRush',
    'teamCode', 'homeTeamCode', 'awayTeamCode',
    'shooterName', 'goalieNameForShot',
    'homeSkatersOnIce', 'awaySkatersOnIce',
    'period'
]

flyers_shot_map = shots_against[shot_map_cols].copy()

#xGA Averages
xga_by_game = shots_against.groupby('game_id').agg(
    total_xGA=('xGoal', 'sum'),
    total_goals_against=('goal', 'sum'),
    total_shots_against=('shotID', 'count'),
    hd_shots=('danger_zone', lambda x: (x == 'High Danger').sum()),
).reset_index().sort_values('game_id').reset_index(drop=True)

xga_by_game['game_num'] = range(1, len(xga_by_game) + 1)
xga_by_game['month_block'] = pd.cut(
    xga_by_game['game_num'],
    bins=[0, 10, 20, 30, 40, 50, 60, 70, 82],
    labels=['Oct', 'Nov', 'Dec', 'Jan', 'Feb', 'Mar Early', 'Mar Late', 'Apr']
)

xga_monthly = xga_by_game.groupby('month_block', observed=True).agg(
    avg_xGA_per_game=('total_xGA', 'mean'),
    avg_shots_against=('total_shots_against', 'mean'),
    avg_hd_shots=('hd_shots', 'mean'),
    games=('game_id', 'count')
).reset_index()

xga_monthly['avg_xGA_per_game'] = xga_monthly['avg_xGA_per_game'].round(2)
xga_monthly['avg_shots_against'] = xga_monthly['avg_shots_against'].round(1)
xga_monthly['avg_hd_shots'] = xga_monthly['avg_hd_shots'].round(1)


#HD shots by Opponet against Flyers
hd_by_opponent = shots_against.groupby('teamCode').agg(
    total_shots=('shotID', 'count'),
    hd_shots=('danger_zone', lambda x: (x == 'High Danger').sum()),
    total_xGA=('xGoal', 'sum'),
    goals=('goal', 'sum')
).reset_index()

hd_by_opponent['hd_shot_pct'] = (hd_by_opponent['hd_shots'] / hd_by_opponent['total_shots'] * 100).round(1)
hd_by_opponent['xGA_per_shot'] = (hd_by_opponent['total_xGA'] / hd_by_opponent['total_shots']).round(4)
hd_by_opponent = hd_by_opponent.sort_values('hd_shot_pct', ascending=False).reset_index(drop=True)



flyers_shot_map.to_csv('/content/drive/MyDrive/Hockey Analytics/flyers_shots_against.csv', index=False)
xga_monthly.to_csv('/content/drive/MyDrive/Hockey Analytics/flyers_xga_monthly.csv', index=False)
hd_by_opponent.to_csv('/content/drive/MyDrive/Hockey Analytics/flyers_hd_by_opponent.csv', index=False)


Total shots loaded: 100744
Shots against Flyers: 2987


In [10]:
#Defensive metrics for the league
all_teams_against = shots.copy()

all_teams_against['defending_team'] = np.where(
    all_teams_against['teamCode'] == all_teams_against['homeTeamCode'],
    all_teams_against['awayTeamCode'],
    all_teams_against['homeTeamCode']
)

all_teams_against['danger_zone'] = all_teams_against['xGoal'].apply(assign_danger_from_xgoal)

team_defense = all_teams_against.groupby('defending_team').agg(
    total_shots_against=('shotID', 'count'),
    total_xGA=('xGoal', 'sum'),
    total_goals_against=('goal', 'sum'),
    hd_shots_against=('danger_zone', lambda x: (x == 'High Danger').sum()),
    md_shots_against=('danger_zone', lambda x: (x == 'Medium Danger').sum()),
    ld_shots_against=('danger_zone', lambda x: (x == 'Low Danger').sum()),
    games=('game_id', 'nunique')
).reset_index()

team_defense['xGA_per_game'] = (team_defense['total_xGA'] / team_defense['games']).round(2)
team_defense['shots_against_per_game'] = (team_defense['total_shots_against'] / team_defense['games']).round(1)
team_defense['hd_shot_pct'] = (team_defense['hd_shots_against'] / team_defense['total_shots_against'] * 100).round(1)
team_defense['md_shot_pct'] = (team_defense['md_shots_against'] / team_defense['total_shots_against'] * 100).round(1)
team_defense['ld_shot_pct'] = (team_defense['ld_shots_against'] / team_defense['total_shots_against'] * 100).round(1)
team_defense['is_flyer'] = (team_defense['defending_team'] == 'PHI').astype(int)

team_defense.to_csv('/content/drive/MyDrive/Hockey Analytics/flyers_vs_league_defense.csv', index=False)

print(team_defense.to_string())

   defending_team  total_shots_against   total_xGA  total_goals_against  hd_shots_against  md_shots_against  ld_shots_against  games  xGA_per_game  shots_against_per_game  hd_shot_pct  md_shot_pct  ld_shot_pct  is_flyer
0             ANA                 3327  252.954244                  256               248               851              2228     74          3.42                    45.0          7.5         25.6         67.0         0
1             BOS                 3383  249.911709                  227               274               832              2277     74          3.38                    45.7          8.1         24.6         67.3         0
2             BUF                 3309  238.890535                  218               261               746              2302     74          3.23                    44.7          7.9         22.5         69.6         0
3             CAR                 2636  208.107588                  213               230               671             